# INF201 Deliverable 2: Campus Coffee Shop Sales Pipeline

GitHub repository: https://github.com/Sivert6/inf201-h26-deliverable-2


In [7]:
import re
from pathlib import Path

DATA_DIR = Path("data")

---

## Task 1: File ingestion and regular expressions

The log files use either UTF-8 or ISO-8859-1 encoding. Our function first
tries UTF-8. If that fails, it uses ISO-8859-1 instead. ISO-8859-1 can read any file, so it will always works as a backup.

In [8]:
def read_log_file(file_path):
    """Read a file. Try UTF-8 first, then ISO-8859-1."""
    path = Path(file_path)
    try:
        text = path.read_text(encoding="utf-8")
        return text, "utf-8"
    except UnicodeDecodeError:
        text = path.read_text(encoding="iso-8859-1")
        return text, "iso-8859-1"

In [9]:
test_file = DATA_DIR / "2026" / "library_cafe" / "POS_01" / "register_sales.log"

text, encoding = read_log_file(test_file)

print("Encoding:", encoding)
print(text[:300])

Encoding: utf-8
[2026-10-15 07:00:14] [POS_01] item=Sandwich price=7.55 USD (SUCCESS)
[2026-10-15 07:00:50] [POS_01] item=Croissant price=4.26 EUR (CANCELLED)
[2026-10-15 07:02:34] [POS_01] item=Cappuccino price=4.59 USD (CANCELLED)
[2026-10-15 07:04:07] [POS_01] item=Espresso price=2.91 NOK (CANCELLED)
[2026-10-15


In [10]:
LOG_PATTERN = (
    r"\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] "
    r"\[(\w+)\] "
    r"item=(\w+) "
    r"price=(\d+\.\d+) "
    r"([A-Z]{3}) "
    r"\((SUCCESS|REFUND|CANCELLED)\)"
)

In [11]:
def parse_log(text):
    """Find all valid sales lines and return them as a list of dictionaries."""
    records = []
    for line in text.splitlines():
        match = re.match(LOG_PATTERN, line)
        if match:
            record = {
                "timestamp": match.group(1),
                "register_id": match.group(2),
                "item": match.group(3),
                "price": float(match.group(4)),
                "currency": match.group(5),
                "status": match.group(6),
            }
            records.append(record)
    return records

In [12]:
records = parse_log(text)

print("Number of valid records:", len(records))
print(records[0])

Number of valid records: 80
{'timestamp': '2026-10-15 07:00:14', 'register_id': 'POS_01', 'item': 'Sandwich', 'price': 7.55, 'currency': 'USD', 'status': 'SUCCESS'}


### Discussion of Task 1

 Function (read_log_file) first tries to read the file as UTF-8. If this gives a UnicodeDecodeError, it reads the file again as ISO-8859-1. This way the function finds the encoding by itself.

Function (parse_log) checks every line against a regular expression with six groups, one for each field. Lines that do not match, such as
--- PRINTER PAPER LOW ---, are skipped. The test file gave 80 valid records, which matches the 80 sales lines the data generator writes per file.